# Agent的高级用法

## 1、设置Agent的名称

经典使用场景:

name在 Multi‑Agent 场景里被频繁使用，主要用来区分不同 Agent，但它的作用不只限于多 Agent 编排。实际工程里，遇到下面这些场景，都建议给 Agent 配置清晰且固定的 name：

1、流式输出归因

开启流式输出时，依靠 name 辨别输出内容归属的 Agent。在多 Agent 协同、Agent 嵌套调用，或是前端需要实时区分不同主体输出内容时十分关键，可以精准分辨 token 和事件的来源。

2、消息身份标记

配置 name 之后，Agent 生成的 AIMessage 会附带对应的 name。系统持久化会话记录、复现执行流程、生成审计日志以及前端渲染消息角色时，依靠该字段确认消息是谁产生的。

3、调试和链路追踪（trace）优化

进行代码调试、日志查看、链路追踪时，name 充当 Agent 的固定标识，方便开发人员快速定位正在运行的 Agent。当系统内部有多个功能近似的 Agent，或是 Agent 嵌套在复杂工作流里，合理命名能够大幅提高链路日志可读性，加快故障排查速度。

4、组件化封装

工程开发中 Agent 会封装成可复用单元，例如检索助手、SQL 助手、报告生成助手。设置 name，在模块注册、运行监控、日志归档、复用组件期间维持统一身份；后续把 Agent 当作子图节点、工具函数、子模块接入大型系统，也会减少维护和迁移的代价。

5、前端展示以及运行状态观测

可视化项目中，name 直接作为前端展示名称。执行面板展示活跃 Agent、消息来源、调用链路节点时，不管是开发人员还是普通用户，都能清晰了解系统实时运行情况。

6、充当稳定的运行时身份标识

宽泛来讲，name 就是 Agent 在系统内部的运行时身份 ID。相较于临时别名，规范稳定的 name 适配日志检索、监控指标统计、链路分析、跨模块协作等场景。生产环境里最好手动指定 name，不要直接使用默认配置。


使用字段name来进行设置

举例：

In [7]:
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os
from langchain.agents import create_agent
from rich import print as rprint

load_dotenv(override=True)

model = init_chat_model(
    model="qwen3.7-plus",
    model_provider="openai",
    api_key=os.getenv("DASHSCOPE_API_KEY"),
    base_url=os.getenv("DASHSCOPE_BASE_URL"),
    extra_body={"enable_thinking": False},
)

agent = create_agent(
    model = model,
    name = "agent01"
)

# 调用
response = agent.invoke({
    "messages":[
        # {"role":"system", "content":"你是一个精通数学的老师，擅长以通俗易懂的方式讲解数学问题"},
        {"role":"user", "content":"100 + 20 * 3 = ？"}
    ]
})

# rprint(response)

for message in response["messages"]:
    message.pretty_print()

================================ Human Message =================================

100 + 20 * 3 = ？
================================== Ai Message ==================================
Name: agent01

根据数学运算的优先级规则（先乘除，后加减），计算步骤如下：

1.  **先计算乘法**：
    $$20 \times 3 = 60$$

2.  **再计算加法**：
    $$100 + 60 = 160$$

因此，最终答案是 **160**。


## 2、系统提示词的设置

使用create_agent创建Agent时，需传入模型和工具、可选地传入系统提示词。提示词为Agent提供了任务背景、行为准则和操作指南。

系统指令，即SystemMessage，通过 system_prompt 设置，定义Agent行为。这个参数可以是str或者SystemMessage类型。

使用建议：

明确说明 Agent 的角色

定义输出格式

说明何时使用工具


使用system_prompt参数进行设置，可以是str，也可以是SystemMessage

提示词设置有两种方式：静态设置和动态设置。动态设置需要借助中间件，后续讲解。

举例1：

In [9]:
from langchain_tavily import TavilySearch
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os
from rich import print as rprint

load_dotenv(override=True)

# 1.导入模型
model = init_chat_model(
    model="qwen3.7-plus",
    model_provider="openai",
    api_key=os.getenv("DASHSCOPE_API_KEY"),
    base_url=os.getenv("DASHSCOPE_BASE_URL"),
    # extra_body={"enable_thinking": False},
)

# 2.导入工具
web_search = TavilySearch(max_results=2)

# 3.创建Agent
agent = create_agent(
    model=model,
    tools=[web_search],
    system_prompt="你是一名多才多艺的智能助手，可以调用工具帮助用户解决问题。"
)

# 4.运行Agent获得结果
result = agent.invoke(
 {"messages": [
     {"role": "user", "content": "请帮我查询2026年足球世界杯是哪个国家举办的？"}
 ]}
)

# print(result['messages'][-1].content)
rprint(result)

{
    'messages': [
        HumanMessage(
            content='请帮我查询2026年足球世界杯是哪个国家举办的？',
            additional_kwargs={},
            response_metadata={},
            id='8d262c12-ea50-4605-afc6-5e3ceb6d9c55'
        ),
        AIMessage(
            content='',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 101,
                    'prompt_tokens': 1837,
                    'total_tokens': 1938,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': None,
                        'audio_tokens': None,
                        'reasoning_tokens': 62,
                        'rejected_prediction_tokens': None,
                        'text_tokens': 101
                    },
                    'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0, 'text_tokens': 1837}
                },
                'model_provider': 'openai',
                'model_name': 'qwen3.7-plus',
                'system_fingerprint': None,
                'id': 'chatcmpl-bb87b0a3-1a8d-9bcc-b4b9-fd541c204645',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--019f65a7-d644-7663-83c0-8b0ff57c3abb-0',
            tool_calls=[
                {
                    'name': 'tavily_search',
                    'args': {'query': '2026年足球世界杯举办国家'},
                    'id': 'call_56c4ddf00ab44706893dde61',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 1837,
                'output_tokens': 101,
                'total_tokens': 1938,
                'input_token_details': {'cache_read': 0},
                'output_token_details': {'reasoning': 62}
            }
        ),
        ToolMessage(
            content='{"query": "2026年足球世界杯举办国家", "follow_up_questions": null, "answer": null, "images": 
[], "results": [{"url": "https://www.xinhuanet.com/20260509/59361d08fa3644beb6bea55caddf3b30/c.html", "title": 
"2026世界杯美加墨三国将分别举办开幕式", "content": "# 2026世界杯美加墨三国将分别举办开幕式. 2026-05-09 16:50:21  
来源：新华网. 
新华社华盛顿5月8日电（记者杨伶）国际足联8日宣布，2026年世界杯举办国墨西哥、加拿大和美国将分别举办开幕式。. 
图为3月28日拍摄的墨西哥城阿兹特克体育场（无人机照片）。新华社记者 李木子 摄. 
国际足联主席因凡蒂诺表示：“从墨西哥城开始，接下来是多伦多和洛杉矶，这些开幕式将音乐、文化和足球融为一体，既体现了每
个国家的独特性，也展现了本届赛事所代表的团结精神……这将是开启一场全球盛会的有力方式。”. 
美加墨世界杯将于6月11日开幕，揭幕战是墨西哥队对阵南非队，比赛将在墨西哥城举行。国际足联在声明中说，比赛开始前90分钟
，墨西哥将以一场“史无前例的开幕式拉开这场地球上最盛大演出的序幕”。加拿大将于6月12日在多伦多举办开幕式，之后加拿大队
将迎战波黑队。美国将于6月12日在洛杉矶举行开幕式，之后美国队将迎来与巴拉圭队的小组赛。. 
美加墨世界杯将于6月11日至7月19日在美国、加拿大、墨西哥的16座城市举行，这是首次有48支球队参赛的世界杯，比赛场次也大
幅增加至104场。.", "score": 0.8687491, "raw_content": null}, {"url": 
"https://www.aps.dz/cn/sports/sport-collectif/mnfzpl39-2026%E5%B9%B4%E4%B8%96%E7%95%8C%E6%9D%AF48%E6%94%AF%E5%8F%82
%E8%B5%9B%E5%9B%BD%E5%AE%B6", "title": "2026年世界杯48支参赛国家", "content": "# 2026年世界杯48支参赛国家. 
巴黎（法国）——伊拉克于周三获得了2026年世界杯的最后一个参赛资格（6月11日—7月19日），这是首次将参赛队伍扩充至48支，并
由三个国家（美国、加拿大和墨西哥）联合举办的世界杯。.", "score": 0.8405867, "raw_content": null}], "response_time":
0.0, "request_id": "f8b67043-5ab1-4fb4-9558-146e3ef64847"}',
            name='tavily_search',
            id='d4c98c76-2764-4784-a106-5bb779174e64',
            tool_call_id='call_56c4ddf00ab44706893dde61'
        ),
        AIMessage(
            content='2026年足球世界杯由**美国、加拿大和墨西哥**三个国家联合举办。\n\n这是历史上首次由三个国家共同举
办的世界杯，也是首次将参赛队伍扩充至48支的世界杯。比赛将于2026年6月11日至7月19日在这三个国家的16座城市举行，比赛场
次也大幅增加至104场。\n\n值得一提的是，三个举办国还将分别举办开幕式：\n- 
**墨西哥**：6月11日在墨西哥城举办开幕式，揭幕战为墨西哥队对阵南非队\n- **加拿大**：6月12日在多伦多举办开幕式\n- 
**美国**：6月12日在洛杉矶举办开幕式',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 267,
                    'prompt_tokens': 2590,
                    'total_tokens': 2857,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': None,
      

举例2

In [12]:
from langchain.agents import create_agent
from langchain_core.messages import SystemMessage
from langchain_core.tools import tool
from rich import print as rprint

# 工具：实现两数相加
@tool
def add_numbers(a: int, b: int) -> str:
    """计算并返回两个数的和。"""
    return f"和为：{a + b}"

# 创建客服助手Agent
agent = create_agent(
    model=model,
    tools=[add_numbers],  # 工具列表
    # system_prompt="你是一个数学助手，解决日常的算术问题"
    system_prompt=SystemMessage(content="你是一个数学助手，解决日常的算术问题,要调用工具")
)

response = agent.invoke({
    "messages": [
     {"role": "user", "content": "10加上20再加上30是多少？"}
 ]
})

rprint(response)
# print(response["messages"][-1].content)

{
    'messages': [
        HumanMessage(
            content='10加上20再加上30是多少？',
            additional_kwargs={},
            response_metadata={},
            id='1c790dc6-deb4-4375-af3c-207e7c0d0926'
        ),
        AIMessage(
            content='',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 295,
                    'prompt_tokens': 304,
                    'total_tokens': 599,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': None,
                        'audio_tokens': None,
                        'reasoning_tokens': 253,
                        'rejected_prediction_tokens': None,
                        'text_tokens': 295
                    },
                    'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0, 'text_tokens': 304}
                },
                'model_provider': 'openai',
                'model_name': 'qwen3.7-plus',
                'system_fingerprint': None,
                'id': 'chatcmpl-e61d3158-830a-9b10-babe-957b02390051',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--019f65b0-5559-7d63-b3da-6b9b5e8052dc-0',
            tool_calls=[
                {
                    'name': 'add_numbers',
                    'args': {'a': 10, 'b': 20},
                    'id': 'call_26b7b953232849e5879e3caf',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 304,
                'output_tokens': 295,
                'total_tokens': 599,
                'input_token_details': {'cache_read': 0},
                'output_token_details': {'reasoning': 253}
            }
        ),
        ToolMessage(
            content='和为：30',
            name='add_numbers',
            id='caa995ee-fb0f-4450-81a9-7a86c7bd978b',
            tool_call_id='call_26b7b953232849e5879e3caf'
        ),
        AIMessage(
            content='',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 78,
                    'prompt_tokens': 364,
                    'total_tokens': 442,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': None,
                        'audio_tokens': None,
                        'reasoning_tokens': 36,
                        'rejected_prediction_tokens': None,
                        'text_tokens': 78
                    },
                    'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0, 'text_tokens': 364}
                },
                'model_provider': 'openai',
                'model_name': 'qwen3.7-plus',
                'system_fingerprint': None,
                'id': 'chatcmpl-cb8bdf67-30cb-9f86-a009-8d6c9d90ac10',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--019f65b0-6d24-7641-bd55-defb3a2391e4-0',
            tool_calls=[
                {
                    'name': 'add_numbers',
                    'args': {'a': 30, 'b': 30},
                    'id': 'call_47d6b8569cb449268828240c',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 364,
                'output_tokens': 78,
                'total_tokens': 442,
                'input_token_details': {'cache_read': 0},
                'output_token_details': {'reasoning': 36}
            }
        ),
        ToolMessage(
            content='和为：60',
            name='add_numbers',
            id='620547c5-9807-4bb9-becf-23e21e3616a2',
            tool_call_id='call_47d6b8569cb449268828240c'
    

举例3：

In [13]:
from langchain.agents import create_agent
from langchain.tools import tool
from langchain.messages import SystemMessage, HumanMessage
from rich import print as rprint

flag = 0

@tool(parse_docstring=True)
def get_weather(city: str):
    """
    天气查询工具

    Args:
        city: 城市名称
    """
    global flag
    flag += 1
    if flag < 3:
        # raise Exception("暂时无法访问")
        return "TEMP_UNAVAILABLE: 天气服务暂时不可用，请稍后重试"
    return f"{city}今天天气挺好"

messages = [
    HumanMessage("你好，杭州今天的天气如何？")
]

agent = create_agent(
    model=model,
    tools=[get_weather],
    system_prompt=SystemMessage(
        "你是一个天气助手。"
        "当工具返回以 'TEMP_UNAVAILABLE:' 开头的结果时，"
        "说明是临时故障，不要立即放弃；"
        "你应再次调用同一个工具，最多重试 3 次。"
        "如果 3 次后仍失败，再向用户说明服务暂时不可用。"
    )
)

response = agent.invoke({"messages": messages})

rprint(response)
# for msg in response["messages"]:
#     msg.pretty_print()

{
    'messages': [
        HumanMessage(
            content='你好，杭州今天的天气如何？',
            additional_kwargs={},
            response_metadata={},
            id='6e28b98b-a8a5-4c3b-9a15-5bf910a558db'
        ),
        AIMessage(
            content='',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 65,
                    'prompt_tokens': 339,
                    'total_tokens': 404,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': None,
                        'audio_tokens': None,
                        'reasoning_tokens': 35,
                        'rejected_prediction_tokens': None,
                        'text_tokens': 65
                    },
                    'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0, 'text_tokens': 339}
                },
                'model_provider': 'openai',
                'model_name': 'qwen3.7-plus',
                'system_fingerprint': None,
                'id': 'chatcmpl-6ee15996-d7c7-968a-ba21-195cc94cd431',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--019f65b3-55a6-7030-a82c-e28bb85e22ff-0',
            tool_calls=[
                {
                    'name': 'get_weather',
                    'args': {'city': '杭州'},
                    'id': 'call_f909c964638d425290001669',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 339,
                'output_tokens': 65,
                'total_tokens': 404,
                'input_token_details': {'cache_read': 0},
                'output_token_details': {'reasoning': 35}
            }
        ),
        ToolMessage(
            content='TEMP_UNAVAILABLE: 天气服务暂时不可用，请稍后重试',
            name='get_weather',
            id='fb1ed009-0dab-476d-9500-5707e8257c91',
            tool_call_id='call_f909c964638d425290001669'
        ),
        AIMessage(
            content='服务出现了临时故障，让我再试一次...\n\n',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 81,
                    'prompt_tokens': 396,
                    'total_tokens': 477,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': None,
                        'audio_tokens': None,
                        'reasoning_tokens': 40,
                        'rejected_prediction_tokens': None,
                        'text_tokens': 81
                    },
                    'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0, 'text_tokens': 396}
                },
                'model_provider': 'openai',
                'model_name': 'qwen3.7-plus',
                'system_fingerprint': None,
                'id': 'chatcmpl-0a85ffa7-168d-9f39-a74d-38067975a9ff',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--019f65b3-5e77-7c21-a95c-085fcb676d27-0',
            tool_calls=[
                {
                    'name': 'get_weather',
                    'args': {'city': '杭州'},
                    'id': 'call_0d1d9cdf8a5d4d31ba2d2147',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 396,
                'output_tokens': 81,
                'total_tokens': 477,
                'input_token_details': {'cache_read': 0},
                'output_token_details': {'reasoning': 40}
            }
        ),
        ToolMessage(
            content='TEMP_UNAVAILABLE: 天气服务暂时不可用，请稍后重试',
            name='get_weather',
            id='04b7d9a3-1c30-47e2-b245-90c5cfec3c71

## 3、结构化输出的4种策略


### 3.1 ProviderStrategy策略：

使用模型提供商的原生结构化输出功能实现结构化输出。

这里所说的“原生结构化输出”指的是大语言模型（LLM）提供商通过其API直接提供的、在模型响应阶段就强制保证 输出格式符合预定规范 的能力，这种能力能够在模型生成内容的源头确保结构化准确性。

适用于支持原生结构化输出的模型，比如OpenAI、Anthropic Claude或xAI Grok等。


In [14]:
from pydantic import BaseModel, Field
from langchain.agents import create_agent
from langchain.agents.structured_output import ProviderStrategy
from langchain.messages import HumanMessage
from rich import print as rprint
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os

load_dotenv(override=True)

# 1、初始化大模型
model = init_chat_model(
    model="qwen3.7-plus",
    model_provider="openai",
    api_key=os.getenv("DASHSCOPE_API_KEY"),
    base_url=os.getenv("DASHSCOPE_BASE_URL"),
    extra_body={"enable_thinking": False},
)

# 2.使用pydantic结构化方式定义
class ContractInfo(BaseModel):
    """用户的联系方式"""
    name : str = Field(description="用户的名称")
    email : str = Field(description="用户的邮箱")
    phone : str = Field(description="用户的电话")

agent = create_agent(
    model=model,
    response_format=ProviderStrategy(ContractInfo)
)

response = agent.invoke({
    "messages": [
        {"role":"user", "content":"从以下信息中提取用户信息：小明的邮箱是lbwnbnb@163.com，电话是13000006666"}
    ]
})

# rprint(response)

{
    'messages': [
        HumanMessage(
            content='从以下信息中提取用户信息：小明的邮箱是lbwnbnb@163.com，电话是13000006666',
            additional_kwargs={},
            response_metadata={},
            id='0b6f8dd1-b49c-42b9-bebf-3cbdd56dff46'
        ),
        AIMessage(
            content='{"name": "小明", "email": "lbwnbnb@163.com", "phone": "13000006666"}',
            additional_kwargs={'parsed': None, 'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 35,
                    'prompt_tokens': 164,
                    'total_tokens': 199,
                    'completion_tokens_details': None,
                    'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0, 'text_tokens': 164}
                },
                'model_provider': 'openai',
                'model_name': 'qwen3.7-plus',
                'system_fingerprint': None,
                'id': 'chatcmpl-6b98b41f-1856-9575-815d-58a8f6ab2f4b',
                'finish_reason': 'stop',
                'logprobs': None
            },
            id='lc_run--019f65c2-0833-74b2-89c3-be1be2205c9a-0',
            tool_calls=[],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 164,
                'output_tokens': 35,
                'total_tokens': 199,
                'input_token_details': {'cache_read': 0},
                'output_token_details': {}
            }
        )
    ],
    'structured_response': ContractInfo(name='小明', email='lbwnbnb@163.com', phone='13000006666')
}

### 3.2 ToolStrategy策略  （推荐使用）

对于不支持原生结构化输出的模型，LangChain采用“ToolStrategy”工具调用的方式实现结构化输出。

此策略兼容绝大多数支持工具调用的现代模型，其核心原理是动态创建一个"虚拟工具"，该工具的输入参数对应着期望的数据结构。

当模型需要生成最终答案时，系统会引导模型"调用"这个虚拟工具，从而间接产生符合要求的结构化数据。


In [20]:
from pydantic import BaseModel, Field
from langchain.agents import create_agent
from langchain.agents.structured_output import ProviderStrategy, ToolStrategy
from langchain.messages import HumanMessage
from rich import print as rprint
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os

load_dotenv(override=True)

# 1、初始化大模型
model = init_chat_model(
    model="qwen3.7-plus",
    model_provider="openai",
    api_key=os.getenv("DASHSCOPE_API_KEY"),
    base_url=os.getenv("DASHSCOPE_BASE_URL"),
    extra_body={"enable_thinking": False},
)

# 2.使用pydantic结构化方式定义
class ContractInfo(BaseModel):
    """用户的联系方式"""
    name : str = Field(description="用户的名称")
    email : str = Field(description="用户的邮箱")
    phone : str = Field(description="用户的电话")

agent = create_agent(
    model=model,
    response_format=ToolStrategy(ContractInfo),
    system_prompt="你是一个用户信息整理助手，负责整理用户信息"

)

response = agent.invoke({
    "messages": [
        # {"role":"user", "content":"从以下信息中提取用户信息：小明的邮箱是lbwnbnb@163.com，电话是13000006666"}
        HumanMessage(content="从以下信息中提取用户信息：小明的邮箱是lbwnbnb@163.com，电话是13000006666")
    ]
})

# rprint(response)
print(response["structured_response"])

name='小明' email='lbwnbnb@163.com' phone='13000006666'


### 3.3 AutoStrategy / Type 策略

**特别注意：**

在LangChain 1.0及以上版本中，直接传递模式（如response_format=ContactInfo）不再支持，必须显式使用ToolStrategy或ProviderStrategy。（经过测试，目前LangChain 1.2版本还可使用）。


In [23]:
from pydantic import BaseModel, Field
from langchain.agents import create_agent
from langchain.agents.structured_output import ProviderStrategy, ToolStrategy, AutoStrategy
from langchain.messages import HumanMessage
from rich import print as rprint
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os

load_dotenv(override=True)

# 1、初始化大模型
model = init_chat_model(
    model="qwen3.7-plus",
    model_provider="openai",
    api_key=os.getenv("DASHSCOPE_API_KEY"),
    base_url=os.getenv("DASHSCOPE_BASE_URL"),
    extra_body={"enable_thinking": False},
)

# 2.使用pydantic结构化方式定义
class ContractInfo(BaseModel):
    """用户的联系方式"""
    name : str = Field(description="用户的名称")
    email : str = Field(description="用户的邮箱")
    phone : str = Field(description="用户的电话")

agent = create_agent(
    model=model,
    # response_format=AutoStrategy(ContractInfo),  # 不推荐使用
    response_format=ContractInfo,  # 不推荐使用
    system_prompt="你是一个用户信息整理助手，负责整理用户信息"

)

response = agent.invoke({
    "messages": [
        # {"role":"user", "content":"从以下信息中提取用户信息：小明的邮箱是lbwnbnb@163.com，电话是13000006666"}
        HumanMessage(content="从以下信息中提取用户信息：小明的邮箱是lbwnbnb@163.com，电话是13000006666")
    ]
})

rprint(response)
# print(response["structured_response"])

{
    'messages': [
        HumanMessage(
            content='从以下信息中提取用户信息：小明的邮箱是lbwnbnb@163.com，电话是13000006666',
            additional_kwargs={},
            response_metadata={},
            id='8735c463-7a4f-4691-8eaa-48111a5a1515'
        ),
        AIMessage(
            content='',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 62,
                    'prompt_tokens': 358,
                    'total_tokens': 420,
                    'completion_tokens_details': None,
                    'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0, 'text_tokens': 358}
                },
                'model_provider': 'openai',
                'model_name': 'qwen3.7-plus',
                'system_fingerprint': None,
                'id': 'chatcmpl-1c475c81-d09e-972d-b241-8a99b68c0f5a',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--019f65d8-b3ff-7d70-aa89-5b32284a0d94-0',
            tool_calls=[
                {
                    'name': 'ContractInfo',
                    'args': {'name': '小明', 'email': 'lbwnbnb@163.com', 'phone': '13000006666'},
                    'id': 'call_d8ab14dea84c491ba3cd5684',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 358,
                'output_tokens': 62,
                'total_tokens': 420,
                'input_token_details': {'cache_read': 0},
                'output_token_details': {}
            }
        ),
        ToolMessage(
            content="Returning structured response: name='小明' email='lbwnbnb@163.com' phone='13000006666'",
            name='ContractInfo',
            id='874f8838-a5e0-4903-87f9-a15e91635b12',
            tool_call_id='call_d8ab14dea84c491ba3cd5684'
        )
    ],
    'structured_response': ContractInfo(name='小明', email='lbwnbnb@163.com', phone='13000006666')
}

### 3.4 None策略

默认配置，表示不以结构化输出，以自然语言响应用户问题。

In [25]:
from pydantic import BaseModel, Field
from langchain.agents import create_agent
from langchain.agents.structured_output import ProviderStrategy, ToolStrategy, AutoStrategy
from langchain.messages import HumanMessage
from rich import print as rprint
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os

load_dotenv(override=True)

# 1、初始化大模型
model = init_chat_model(
    model="qwen3.7-plus",
    model_provider="openai",
    api_key=os.getenv("DASHSCOPE_API_KEY"),
    base_url=os.getenv("DASHSCOPE_BASE_URL"),
    extra_body={"enable_thinking": False},
)

# 2.使用pydantic结构化方式定义
class ContractInfo(BaseModel):
    """用户的联系方式"""
    name : str = Field(description="用户的名称")
    email : str = Field(description="用户的邮箱")
    phone : str = Field(description="用户的电话")

agent = create_agent(
    model=model,
    response_format=None

)

response = agent.invoke({
    "messages": [
        # {"role":"user", "content":"从以下信息中提取用户信息：小明的邮箱是lbwnbnb@163.com，电话是13000006666"}
        HumanMessage(content="从以下信息中提取用户信息：小明的邮箱是lbwnbnb@163.com，电话是13000006666")
    ]
})

# rprint(response)
# print(response["structured_response"])
print(response["messages"][-1].content)

从提供的信息中提取的用户信息如下：

*   **姓名**：小明
*   **邮箱**：lbwnbnb@163.com
*   **电话**：13000006666


**总结：**

在实际大模型Agent开发场景中，如果使用到了结构化输出，推荐使用 “**ToolStrategy**”策略 ，所以后续重点介绍这种策略方式结构化输出。
